# Autonomous Vehicle Accident Prediction
**Binary Classification — Random Forest + MLP Neural Network Ensemble**

This notebook:
1. Generates a synthetic AV sensor dataset
2. Preprocesses and engineers features
3. Trains a Random Forest and an MLP Neural Network
4. Evaluates both models and the ensemble
5. Saves all model files to disk

## 1. Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install numpy pandas scikit-learn tensorflow imbalanced-learn matplotlib seaborn joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score, accuracy_score)
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow version:', tf.__version__)
print('All libraries loaded successfully.')

## 2. Generate Synthetic AV Sensor Dataset

In [ ]:
def generate_av_dataset(n_samples=12000, random_state=42):
    """
    Generate a synthetic dataset simulating autonomous vehicle sensor readings.
    Each row is one driving episode. Label=1 means an accident occurred.
    """
    rng = np.random.RandomState(random_state)

    # --- Raw sensor features ---
    speed_kmh          = rng.uniform(0, 180, n_samples)
    following_dist_m   = rng.uniform(1, 80, n_samples)
    visibility_m       = rng.uniform(10, 300, n_samples)
    road_wetness       = rng.uniform(0, 1, n_samples)
    pedestrian_count   = rng.randint(0, 21, n_samples).astype(float)
    lane_change_rate   = rng.uniform(0, 15, n_samples)
    sensor_health      = rng.uniform(0, 1, n_samples)
    traffic_density    = rng.uniform(0, 10, n_samples)
    steering_deviation = rng.uniform(0, 10, n_samples)

    # --- Categorical features ---
    time_of_day = rng.choice(['day', 'dusk', 'night'], n_samples, p=[0.5, 0.2, 0.3])
    road_type   = rng.choice(['highway', 'urban', 'rural'], n_samples, p=[0.4, 0.4, 0.2])
    weather     = rng.choice(['clear', 'rain', 'fog', 'snow'], n_samples, p=[0.5, 0.25, 0.15, 0.1])

    # --- Accident probability (physics-based label generation) ---
    risk = (
        0.22 * np.clip((speed_kmh - 50) / 130, 0, 1) +
        0.18 * np.clip(1 - following_dist_m / 40, 0, 1) +
        0.16 * np.clip(1 - visibility_m / 250, 0, 1) +
        0.12 * road_wetness +
        0.10 * (1 - sensor_health) +
        0.08 * traffic_density / 10 +
        0.07 * pedestrian_count / 20 +
        0.04 * lane_change_rate / 15 +
        0.03 * steering_deviation / 10 +
        0.02 * (time_of_day == 'night').astype(float) +
        0.02 * (weather == 'fog').astype(float) +
        0.01 * (weather == 'snow').astype(float)
    )
    noise = rng.normal(0, 0.05, n_samples)
    prob  = np.clip(risk + noise, 0, 1)
    label = (prob > 0.45).astype(int)

    df = pd.DataFrame({
        'speed_kmh':          speed_kmh,
        'following_dist_m':   following_dist_m,
        'visibility_m':       visibility_m,
        'road_wetness':       road_wetness,
        'pedestrian_count':   pedestrian_count,
        'lane_change_rate':   lane_change_rate,
        'sensor_health':      sensor_health,
        'traffic_density':    traffic_density,
        'steering_deviation': steering_deviation,
        'time_of_day':        time_of_day,
        'road_type':          road_type,
        'weather':            weather,
        'accident':           label
    })
    return df

df = generate_av_dataset()
print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['accident'].value_counts(normalize=True).rename({0:'No accident', 1:'Accident'}).map('{:.1%}'.format))
df.head()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature distributions by accident class', fontsize=14, fontweight='bold')

numeric_features = ['speed_kmh', 'following_dist_m', 'visibility_m',
                    'road_wetness', 'sensor_health', 'traffic_density']

for ax, feat in zip(axes.flatten(), numeric_features):
    df[df['accident']==0][feat].hist(ax=ax, alpha=0.6, label='No accident', color='#639922', bins=30)
    df[df['accident']==1][feat].hist(ax=ax, alpha=0.6, label='Accident',    color='#E24B4A', bins=30)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

## 4. Feature Engineering & Preprocessing

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Derived safety features
    df['safety_margin']   = df['following_dist_m'] / (df['speed_kmh'] / 3.6 + 1e-6)
    df['visibility_ratio']= df['visibility_m'] / 300
    df['speed_excess']    = np.clip((df['speed_kmh'] - 50) / 130, 0, 1)
    df['composite_risk']  = df['road_wetness'] * (1 / (df['visibility_m'] + 1)) * df['speed_kmh']
    df['sensor_penalty']  = 1 - df['sensor_health']
    df['congestion_idx']  = df['traffic_density'] * df['pedestrian_count']

    # One-hot encode categoricals
    df = pd.get_dummies(df, columns=['time_of_day', 'road_type', 'weather'], drop_first=False)
    return df

df_feat = engineer_features(df)

X = df_feat.drop('accident', axis=1)
y = df_feat['accident']

print('Feature matrix shape:', X.shape)
print('Features:', list(X.columns))

In [ ]:
# Train / Validation / Test split  (70 / 15 / 15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}')

# SMOTE — balance training set only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — Train: {X_train_res.shape[0]}')

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print('Preprocessing complete.')

## 5. Train Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_sc, y_train_res)

rf_val_preds = rf.predict(X_val_sc)
rf_val_proba = rf.predict_proba(X_val_sc)[:, 1]

print('Random Forest — Validation results')
print('Accuracy :', accuracy_score(y_val, rf_val_preds))
print('ROC-AUC  :', roc_auc_score(y_val, rf_val_proba))
print('F1 Score :', f1_score(y_val, rf_val_preds))
print()
print(classification_report(y_val, rf_val_preds, target_names=['No accident','Accident']))

In [ ]:
# Feature importance plot
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
top15 = importances.tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
top15.plot(kind='barh', ax=ax, color='#3266ad')
ax.set_title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
ax.set_xlabel('Gini importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Train MLP Neural Network

In [ ]:
def build_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(32, activation='relu'),
        layers.Dense(1,  activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

mlp = build_mlp(X_train_sc.shape[1])
mlp.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_auc', patience=7, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('best_mlp.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=0)
]

history = mlp.fit(
    X_train_sc, y_train_res,
    validation_data=(X_val_sc, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Training history plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(history.history['accuracy'],     label='Train acc', color='#3266ad', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val acc',   color='#73726c', linewidth=2, linestyle='--')
ax1.set_title('Accuracy over epochs', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history.history['loss'],     label='Train loss', color='#E24B4A', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val loss',   color='#ba7517', linewidth=2, linestyle='--')
ax2.set_title('Loss over epochs', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=120, bbox_inches='tight')
plt.show()

mlp_val_proba = mlp.predict(X_val_sc).flatten()
mlp_val_preds = (mlp_val_proba >= 0.45).astype(int)
print('MLP — Validation results')
print('Accuracy :', accuracy_score(y_val, mlp_val_preds))
print('ROC-AUC  :', roc_auc_score(y_val, mlp_val_proba))
print(classification_report(y_val, mlp_val_preds, target_names=['No accident','Accident']))

## 7. Ensemble — Soft Voting

In [ ]:
THRESHOLD = 0.45  # Lower threshold — safety-critical: prefer catching accidents

def ensemble_predict(X_scaled, threshold=THRESHOLD):
    rf_proba  = rf.predict_proba(X_scaled)[:, 1]
    mlp_proba = mlp.predict(X_scaled, verbose=0).flatten()
    avg_proba = 0.5 * rf_proba + 0.5 * mlp_proba
    preds = (avg_proba >= threshold).astype(int)
    return preds, avg_proba

# Evaluate on test set
y_pred, y_proba = ensemble_predict(X_test_sc)

print('=' * 50)
print('ENSEMBLE — Final Test Set Results')
print('=' * 50)
print(f'Accuracy  : {accuracy_score(y_test, y_pred):.4f}')
print(f'ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}')
print(f'F1 Score  : {f1_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No accident','Accident']))

In [ ]:
# Confusion matrix + ROC curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax1,
            xticklabels=['No accident','Accident'],
            yticklabels=['No accident','Accident'])
ax1.set_title('Confusion Matrix — Test Set', fontweight='bold')
ax1.set_ylabel('Actual'); ax1.set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
ax2.plot(fpr, tpr, color='#3266ad', linewidth=2, label=f'Ensemble (AUC = {auc:.3f})')
ax2.plot([0,1],[0,1], color='#B4B2A9', linestyle='--', linewidth=1)
ax2.set_title('ROC Curve', fontweight='bold')
ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_plots.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Save All Model Files

In [ ]:
os.makedirs('saved_models', exist_ok=True)

# Save Random Forest
joblib.dump(rf, 'saved_models/random_forest.pkl')
print('Saved: saved_models/random_forest.pkl')

# Save MLP
mlp.save('saved_models/mlp_model.keras')
print('Saved: saved_models/mlp_model.keras')

# Save scaler
joblib.dump(scaler, 'saved_models/scaler.pkl')
print('Saved: saved_models/scaler.pkl')

# Save feature column names (needed when loading)
feature_cols = list(X.columns)
joblib.dump(feature_cols, 'saved_models/feature_columns.pkl')
print('Saved: saved_models/feature_columns.pkl')

print('\nAll model files saved to saved_models/')

## 9. Load & Use the Saved Model

In [ ]:
# --- How to reload and predict on new data ---

rf_loaded     = joblib.load('saved_models/random_forest.pkl')
mlp_loaded    = keras.models.load_model('saved_models/mlp_model.keras')
scaler_loaded = joblib.load('saved_models/scaler.pkl')
feat_cols     = joblib.load('saved_models/feature_columns.pkl')

print('All models loaded successfully.')

# Example: predict on a single new driving episode
new_episode = pd.DataFrame([{
    'speed_kmh': 95,
    'following_dist_m': 8,
    'visibility_m': 40,
    'road_wetness': 0.8,
    'pedestrian_count': 5,
    'lane_change_rate': 7,
    'sensor_health': 0.6,
    'traffic_density': 8,
    'steering_deviation': 4,
    'time_of_day': 'night',
    'road_type': 'urban',
    'weather': 'rain'
}])

# Apply same feature engineering
new_feat = engineer_features(new_episode.assign(accident=0)).drop('accident', axis=1)

# Align columns (fill missing one-hot cols with 0)
new_feat = new_feat.reindex(columns=feat_cols, fill_value=0)

# Scale
new_scaled = scaler_loaded.transform(new_feat)

# Ensemble prediction
rf_p  = rf_loaded.predict_proba(new_scaled)[0, 1]
mlp_p = mlp_loaded.predict(new_scaled, verbose=0)[0, 0]
final_p = 0.5 * rf_p + 0.5 * mlp_p
label   = 'ACCIDENT' if final_p >= 0.45 else 'SAFE'

print(f'\nRandom Forest probability : {rf_p:.3f}')
print(f'MLP probability           : {mlp_p:.3f}')
print(f'Ensemble probability      : {final_p:.3f}')
print(f'Prediction                : {label}')